# 📊 Planilha Embriologia Validation: Local DuckDB vs AWS Athena (Prod)

This notebook automates the validation and comparison of the three versions of `planilha_embriologia` (combined, fresh, and fet) between the local DuckDB database (`huntington_data_lake.duckdb`) and production AWS Athena (`silver_embriologia_prod`).

### Rules:
- **Rule**: Each code cell performing queries must explicitly open and close database connections.


In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
from pyathena import connect
import warnings
warnings.filterwarnings('ignore')

# Database Configurations
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_embriologia_prod'

print("Configuration set.")
print(f"Local DuckDB Path: {os.path.abspath(DUCKDB_PATH)}")
print(f"AWS Athena Schema: {ATHENA_DB}")


In [ ]:
# Database Configuration
DUCKDB_PATH = 'database/huntington_data_lake.duckdb'
ATHENA_REGION = 'sa-east-1'
ATHENA_WORKGROUP = 'datalake-admins'
ATHENA_DB = 'silver_embriologia_prod'


In [ ]:
try:
    with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
        duck_ok = conn.execute("SELECT 1 as test").fetchone()[0] == 1
    print("✅ Local DuckDB Connection: OK")
except Exception as e:
    print(f"❌ Local DuckDB Connection: FAILED - {e}")
    duck_ok = False

try:
    with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1")
            athena_ok = cur.fetchone()[0] == 1
    print("✅ AWS Athena Connection: OK")
except Exception as e:
    print(f"❌ AWS Athena Connection: FAILED - {e}")
    athena_ok = False


## 🟣 Part 1: Reconcile fresh Table (Local planilha_embriologia_fresh vs Athena fresh)

In this section, we compare schemas, counts, distinct keys, incubator distributions, and clinical metric sums.


In [ ]:
# 1. Compare fresh schemas
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM silver.planilha_embriologia_fresh LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM fresh LIMIT 0", conn).columns]

only_local = sorted(list(set(local_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(local_cols)))
common_cols = sorted(list(set(local_cols) & set(prod_cols)))

print(f"Fresh columns in Local only: {only_local}")
print(f"Fresh columns in Athena only: {only_prod}")
print(f"Common columns: {len(common_cols)}")


In [ ]:
# 2. Compare Counts and Keys for fresh
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_counts = conn.execute('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM silver.planilha_embriologia_fresh
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_counts = pd.read_sql('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM fresh
    ''', conn)

df_counts = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique PIN', 'Unique Prontuario'],
    'Local (DuckDB)': [local_counts.loc[0, 'total_rows'], local_counts.loc[0, 'unique_pin'], local_counts.loc[0, 'unique_prontuario']],
    'Athena (Prod)': [prod_counts.loc[0, 'total_rows'], prod_counts.loc[0, 'unique_pin'], prod_counts.loc[0, 'unique_prontuario']]
})
df_counts['Delta'] = df_counts['Local (DuckDB)'] - df_counts['Athena (Prod)']
df_counts['Match Rate %'] = np.where(
    df_counts['Delta'] == 0,
    100.0,
    (1 - (df_counts['Delta'].abs() / df_counts['Local (DuckDB)'])) * 100
)
display(df_counts)


In [ ]:
# 3. Compare Incubator Distribution for fresh
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_inc = conn.execute('''
        SELECT 
            CASE 
                WHEN incubadora IN ('\\', '-', '/', '.', '0', 'N/A', 'NA', 'none', '') OR incubadora IS NULL THEN 'None / Unspecified'
                WHEN incubadora ILIKE '%THERMO%' THEN 'THERMO'
                WHEN incubadora ILIKE '%ES%' OR incubadora ILIKE '%EMBRYOSCOPE%' THEN 'EMBRYOSCOPE'
                ELSE 'K-SYSTEM'
            END as incubator,
            COUNT(*) as loc_count
        FROM silver.planilha_embriologia_fresh
        GROUP BY 1
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    ath_inc = pd.read_sql('''
        SELECT 
            CASE 
                WHEN incubadora IN ('\\', '-', '/', '.', '0', 'N/A', 'NA', 'none', '') OR incubadora IS NULL THEN 'None / Unspecified'
                WHEN incubadora LIKE '%THERMO%' THEN 'THERMO'
                WHEN incubadora LIKE '%ES%' OR incubadora LIKE '%EMBRYOSCOPE%' THEN 'EMBRYOSCOPE'
                ELSE 'K-SYSTEM'
            END as incubator,
            COUNT(*) as ath_count
        FROM fresh
        GROUP BY 1
    ''', conn)

df_inc_fresh = pd.merge(loc_inc, ath_inc, on='incubator', how='outer').fillna(0)
df_inc_fresh['Delta'] = df_inc_fresh['loc_count'] - df_inc_fresh['ath_count']
df_inc_fresh['Match Rate %'] = np.where(
    df_inc_fresh['Delta'] == 0,
    100.0,
    (1.0 - (df_inc_fresh['Delta'].abs() / df_inc_fresh['loc_count'])) * 100.0
)
df_inc_fresh = df_inc_fresh.sort_values(by='loc_count', ascending=False).reset_index(drop=True)
print('--- Incubator Distribution for fresh ---')
print(df_inc_fresh.to_string(index=False))


In [ ]:
# 4. Compare Clinical Metric Aggregations for fresh (Sum Proofs)
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_outcomes = conn.execute('''
        SELECT 
            SUM(TRY_CAST(qtd_blasto AS DOUBLE)) as sum_qtd_blasto,
            SUM(TRY_CAST(qtd_blasto_tq_a_e_b AS DOUBLE)) as sum_qtd_blasto_tq,
            SUM(TRY_CAST(no_biopsiados AS DOUBLE)) as sum_no_biopsiados,
            SUM(TRY_CAST(qtd_analisados AS DOUBLE)) as sum_qtd_analisados,
            SUM(TRY_CAST(qtd_normais AS DOUBLE)) as sum_qtd_normais,
            SUM(TRY_CAST(total_de_mii AS DOUBLE)) as sum_total_mii,
            SUM(TRY_CAST(opu AS DOUBLE)) as sum_opu
        FROM silver.planilha_embriologia_fresh
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_outcomes = pd.read_sql('''
        SELECT 
            SUM(TRY_CAST(qtd_blasto AS DOUBLE)) as sum_qtd_blasto,
            SUM(TRY_CAST(qtd_blasto_tq AS DOUBLE)) as sum_qtd_blasto_tq,
            SUM(TRY_CAST(n_biopsiados AS DOUBLE)) as sum_no_biopsiados,
            SUM(TRY_CAST(qtd_analisados AS DOUBLE)) as sum_qtd_analisados,
            SUM(TRY_CAST(qtd_normais AS DOUBLE)) as sum_qtd_normais,
            SUM(TRY_CAST(total_de_mii AS DOUBLE)) as sum_total_mii,
            SUM(TRY_CAST(opu AS DOUBLE)) as sum_opu
        FROM fresh
    ''', conn)

fresh_cols = [
    ('Sum qtd_blasto', 'sum_qtd_blasto'),
    ('Sum qtd_blasto_tq', 'sum_qtd_blasto_tq'),
    ('Sum no_biopsiados', 'sum_no_biopsiados'),
    ('Sum qtd_analisados', 'sum_qtd_analisados'),
    ('Sum qtd_normais', 'sum_qtd_normais'),
    ('Sum total_de_mii', 'sum_total_mii'),
    ('Sum opu', 'sum_opu')
]

df_outcomes = pd.DataFrame({
    'Outcome Metric': [c[0] for c in fresh_cols],
    'Local (DuckDB)': [local_outcomes.loc[0, c[1]] for c in fresh_cols],
    'Athena (Prod)': [prod_outcomes.loc[0, c[1]] for c in fresh_cols]
})
df_outcomes['Delta'] = df_outcomes['Local (DuckDB)'] - df_outcomes['Athena (Prod)']
df_outcomes['Match Rate %'] = np.where(
    df_outcomes['Delta'] == 0,
    100.0,
    (1 - (df_outcomes['Delta'].abs() / df_outcomes['Local (DuckDB)'])) * 100
)
print("--- Fresh Clinical Metric Aggregations ---")
display(df_outcomes)


## 🟣 Part 2: Reconcile fet Table (Local planilha_embriologia_fet vs Athena fet)

In this section, we compare schemas, counts, distinct keys, outcome value distributions, and quantitative sums.


In [ ]:
# 1. Compare fet schemas
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM silver.planilha_embriologia_fet LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM fet LIMIT 0", conn).columns]

only_local = sorted(list(set(local_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(local_cols)))
common_cols = sorted(list(set(local_cols) & set(prod_cols)))

print(f"Fet columns in Local only: {only_local}")
print(f"Fet columns in Athena only: {only_prod}")
print(f"Common columns: {len(common_cols)}")


In [ ]:
# 2. Compare Counts and Keys for fet
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_counts = conn.execute('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM silver.planilha_embriologia_fet
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_counts = pd.read_sql('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT pin) as unique_pin,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM fet
    ''', conn)

df_counts = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique PIN', 'Unique Prontuario'],
    'Local (DuckDB)': [local_counts.loc[0, 'total_rows'], local_counts.loc[0, 'unique_pin'], local_counts.loc[0, 'unique_prontuario']],
    'Athena (Prod)': [prod_counts.loc[0, 'total_rows'], prod_counts.loc[0, 'unique_pin'], prod_counts.loc[0, 'unique_prontuario']]
})
df_counts['Delta'] = df_counts['Local (DuckDB)'] - df_counts['Athena (Prod)']
df_counts['Match Rate %'] = np.where(
    df_counts['Delta'] == 0,
    100.0,
    (1 - (df_counts['Delta'].abs() / df_counts['Local (DuckDB)'])) * 100
)
display(df_counts)


In [ ]:
# 3. Compare Outcome Values for fet (distributions of result, gravidez_clinica, gravidez_bioquimica, no_nascidos)
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_res = conn.execute("SELECT result, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY result").df()
    local_gc = conn.execute("SELECT gravidez_clinica, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY gravidez_clinica").df()
    local_gb = conn.execute("SELECT gravidez_bioquimica, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY gravidez_bioquimica").df()
    local_nn = conn.execute("SELECT no_nascidos, COUNT(*) as cnt FROM silver.planilha_embriologia_fet GROUP BY no_nascidos").df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_res = pd.read_sql("SELECT result, COUNT(*) as cnt FROM fet GROUP BY result", conn)
    prod_gc = pd.read_sql("SELECT gravidez_clinica, COUNT(*) as cnt FROM fet GROUP BY gravidez_clinica", conn)
    prod_gb = pd.read_sql("SELECT gravidez_bioquimica, COUNT(*) as cnt FROM fet GROUP BY gravidez_bioquimica", conn)
    prod_nn = pd.read_sql("SELECT n_nascidos as no_nascidos, COUNT(*) as cnt FROM fet GROUP BY n_nascidos", conn)

def clean_category_series(series):
    s = series.fillna('NULL').astype(str).str.strip()
    s = s.str.replace(r'\.0$', '', regex=True)
    s = s.replace({
        'nan': 'NULL', 'None': 'NULL', '<NA>': 'NULL', '': 'NULL',
        'NAN': 'NULL', 'NONE': 'NULL', '<na>': 'NULL', 'none': 'NULL', 'null': 'NULL'
    })
    return s.str.upper()

def merge_dist(local_df, prod_df, col_name):
    local_df = local_df.copy()
    prod_df = prod_df.copy()
    local_df.columns = [col_name, 'Local']
    prod_df.columns = [col_name, 'Athena']
    
    # Normalize nulls/nones and string formats
    local_df[col_name] = clean_category_series(local_df[col_name])
    prod_df[col_name] = clean_category_series(prod_df[col_name])
    
    # Re-group after cleaning
    local_df = local_df.groupby(col_name, as_index=False).sum()
    prod_df = prod_df.groupby(col_name, as_index=False).sum()
    
    merged = pd.merge(local_df, prod_df, on=col_name, how='outer').fillna(0)
    merged['Delta'] = merged['Local'] - merged['Athena']
    merged['Match Rate %'] = np.where(
        merged['Delta'] == 0,
        100.0,
        (1.0 - (merged['Delta'].abs() / merged[['Local', 'Athena']].max(axis=1))) * 100.0
    )
    return merged.sort_values(by='Local', ascending=False).reset_index(drop=True)

print("--- Result Distribution ---")
display(merge_dist(local_res, prod_res, 'result'))

print("\n--- Gravidez Clinica Distribution ---")
display(merge_dist(local_gc, prod_gc, 'gravidez_clinica'))

print("\n--- Gravidez Bioquimica Distribution ---")
display(merge_dist(local_gb, prod_gb, 'gravidez_bioquimica'))

print("\n--- No Nascidos Distribution ---")
display(merge_dist(local_nn, prod_nn, 'no_nascidos'))


In [ ]:
# 4. Compare Quantitative Metric Aggregations for fet
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_fet_sums = conn.execute('''
        SELECT 
            SUM(TRY_CAST(no_nascidos AS DOUBLE)) as sum_no_nascidos,
            SUM(TRY_CAST(no_et AS DOUBLE)) as sum_no_et
        FROM silver.planilha_embriologia_fet
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_fet_sums = pd.read_sql('''
        SELECT 
            SUM(TRY_CAST(n_nascidos AS DOUBLE)) as sum_no_nascidos,
            SUM(TRY_CAST(n_et AS DOUBLE)) as sum_no_et
        FROM fet
    ''', conn)

fet_sum_cols = [
    ('Sum no_nascidos', 'sum_no_nascidos'),
    ('Sum no_et', 'sum_no_et')
]

df_fet_sums = pd.DataFrame({
    'FET Metric': [c[0] for c in fet_sum_cols],
    'Local (DuckDB)': [local_fet_sums.loc[0, c[1]] for c in fet_sum_cols],
    'Athena (Prod)': [prod_fet_sums.loc[0, c[1]] for c in fet_sum_cols]
})
df_fet_sums['Delta'] = df_fet_sums['Local (DuckDB)'] - df_fet_sums['Athena (Prod)']
df_fet_sums['Match Rate %'] = np.where(
    df_fet_sums['Delta'] == 0,
    100.0,
    (1.0 - (df_fet_sums['Delta'].abs() / df_fet_sums['Local (DuckDB)'])) * 100.0
)
print('--- FET Quantitative Metric Aggregations ---')
print(df_fet_sums.to_string(index=False))


## 🟣 Part 3: Reconcile planilha_embriologia_combined (Local vs Athena)

In this section, we compare schemas, counts, unique keys, incubator distributions, fresh clinical sums, FET outcomes, and quantitative proofs for the combined table.


In [ ]:
# 1. Compare combined schemas
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_cols = [col[0].lower() for col in conn.execute("SELECT * FROM silver.planilha_embriologia_combined LIMIT 0").description]

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_cols = [col.lower() for col in pd.read_sql("SELECT * FROM planilha_embriologia_combined LIMIT 0", conn).columns]

only_local = sorted(list(set(local_cols) - set(prod_cols)))
only_prod = sorted(list(set(prod_cols) - set(local_cols)))
common_cols = sorted(list(set(local_cols) & set(prod_cols)))

print(f"Combined columns in Local only: {only_local}")
print(f"Combined columns in Athena only: {only_prod}")
print(f"Common columns: {len(common_cols)}")


In [ ]:
# 2. Compare Counts and Keys for combined
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_counts = conn.execute('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM silver.planilha_embriologia_combined
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_counts = pd.read_sql('''
        SELECT 
            COUNT(*) as total_rows,
            COUNT(DISTINCT prontuario) as unique_prontuario
        FROM planilha_embriologia_combined
    ''', conn)

df_counts = pd.DataFrame({
    'Metric': ['Total Rows', 'Unique Prontuario'],
    'Local (DuckDB)': [local_counts.loc[0, 'total_rows'], local_counts.loc[0, 'unique_prontuario']],
    'Athena (Prod)': [prod_counts.loc[0, 'total_rows'], prod_counts.loc[0, 'unique_prontuario']]
})
df_counts['Delta'] = df_counts['Local (DuckDB)'] - df_counts['Athena (Prod)']
df_counts['Match Rate %'] = np.where(
    df_counts['Delta'] == 0,
    100.0,
    (1 - (df_counts['Delta'].abs() / df_counts['Local (DuckDB)'])) * 100
)
display(df_counts)


In [ ]:
# 3. Compare Incubator Distribution for combined table
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_inc_comb = conn.execute('''
        SELECT 
            CASE 
                WHEN fresh_incubadora IN ('\\', '-', '/', '.', '0', 'N/A', 'NA', 'none', '') OR fresh_incubadora IS NULL THEN 'None / Unspecified'
                WHEN fresh_incubadora ILIKE '%THERMO%' THEN 'THERMO'
                WHEN fresh_incubadora ILIKE '%ES%' OR fresh_incubadora ILIKE '%EMBRYOSCOPE%' THEN 'EMBRYOSCOPE'
                ELSE 'K-SYSTEM'
            END as incubator,
            COUNT(*) as loc_count
        FROM silver.planilha_embriologia_combined
        GROUP BY 1
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    ath_inc_comb = pd.read_sql('''
        SELECT 
            CASE 
                WHEN fresh_incubadora IN ('\\', '-', '/', '.', '0', 'N/A', 'NA', 'none', '') OR fresh_incubadora IS NULL THEN 'None / Unspecified'
                WHEN fresh_incubadora LIKE '%THERMO%' THEN 'THERMO'
                WHEN fresh_incubadora LIKE '%ES%' OR fresh_incubadora LIKE '%EMBRYOSCOPE%' THEN 'EMBRYOSCOPE'
                ELSE 'K-SYSTEM'
            END as incubator,
            COUNT(*) as ath_count
        FROM planilha_embriologia_combined
        GROUP BY 1
    ''', conn)

df_inc_comb = pd.merge(loc_inc_comb, ath_inc_comb, on='incubator', how='outer').fillna(0)
df_inc_comb['Delta'] = df_inc_comb['loc_count'] - df_inc_comb['ath_count']
df_inc_comb['Match Rate %'] = np.where(
    df_inc_comb['Delta'] == 0,
    100.0,
    (1.0 - (df_inc_comb['Delta'].abs() / df_inc_comb['loc_count'])) * 100.0
)
df_inc_comb = df_inc_comb.sort_values(by='loc_count', ascending=False).reset_index(drop=True)
print('--- Incubator Distribution for combined ---')
print(df_inc_comb.to_string(index=False))


In [ ]:
# 4. Compare Fresh Clinical Metric Aggregations for combined table
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    local_outcomes = conn.execute('''
        SELECT 
            SUM(TRY_CAST(fresh_qtd_blasto AS DOUBLE)) as sum_fresh_qtd_blasto,
            SUM(TRY_CAST(fresh_qtd_blasto_tq_a_e_b AS DOUBLE)) as sum_fresh_qtd_blasto_tq,
            SUM(TRY_CAST(fresh_no_biopsiados AS DOUBLE)) as sum_fresh_no_biopsiados,
            SUM(TRY_CAST(fresh_qtd_analisados AS DOUBLE)) as sum_fresh_qtd_analisados,
            SUM(TRY_CAST(fresh_qtd_normais AS DOUBLE)) as sum_fresh_qtd_normais,
            SUM(TRY_CAST(fresh_total_de_mii AS DOUBLE)) as sum_fresh_total_mii,
            SUM(TRY_CAST(fresh_opu AS DOUBLE)) as sum_fresh_opu
        FROM silver.planilha_embriologia_combined
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    prod_outcomes = pd.read_sql('''
        SELECT 
            SUM(TRY_CAST(fresh_qtd_blasto AS DOUBLE)) as sum_fresh_qtd_blasto,
            SUM(TRY_CAST(fresh_qtd_blasto_tq AS DOUBLE)) as sum_fresh_qtd_blasto_tq,
            SUM(TRY_CAST(fresh_n_biopsiados AS DOUBLE)) as sum_fresh_no_biopsiados,
            SUM(TRY_CAST(fresh_qtd_analisados AS DOUBLE)) as sum_fresh_qtd_analisados,
            SUM(TRY_CAST(fresh_qtd_normais AS DOUBLE)) as sum_fresh_qtd_normais,
            SUM(TRY_CAST(fresh_total_de_mii AS DOUBLE)) as sum_fresh_total_mii,
            SUM(TRY_CAST(fresh_opu AS DOUBLE)) as sum_fresh_opu
        FROM planilha_embriologia_combined
    ''', conn)

fresh_comb_cols = [
    ('Sum fresh_qtd_blasto', 'sum_fresh_qtd_blasto'),
    ('Sum fresh_qtd_blasto_tq', 'sum_fresh_qtd_blasto_tq'),
    ('Sum fresh_no_biopsiados', 'sum_fresh_no_biopsiados'),
    ('Sum fresh_qtd_analisados', 'sum_fresh_qtd_analisados'),
    ('Sum fresh_qtd_normais', 'sum_fresh_qtd_normais'),
    ('Sum fresh_total_de_mii', 'sum_fresh_total_mii'),
    ('Sum fresh_opu', 'sum_fresh_opu')
]

df_comb_fresh = pd.DataFrame({
    'Fresh Outcome Metric': [c[0] for c in fresh_comb_cols],
    'Local (DuckDB)': [local_outcomes.loc[0, c[1]] for c in fresh_comb_cols],
    'Athena (Prod)': [prod_outcomes.loc[0, c[1]] for c in fresh_comb_cols]
})
df_comb_fresh['Delta'] = df_comb_fresh['Local (DuckDB)'] - df_comb_fresh['Athena (Prod)']
df_comb_fresh['Match Rate %'] = np.where(
    df_comb_fresh['Delta'] == 0,
    100.0,
    (1 - (df_comb_fresh['Delta'].abs() / df_comb_fresh['Local (DuckDB)'])) * 100
)
print("--- Fresh Outcome Metric Aggregations (Combined) ---")
display(df_comb_fresh)


In [ ]:
# 5. Compare FET Outcome Distributions in combined table
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_comb_res = conn.execute("SELECT fet_resultado as resultado, COUNT(*) as cnt FROM silver.planilha_embriologia_combined GROUP BY 1").df()
    loc_comb_gc = conn.execute("SELECT fet_gravidez_clinica as gravidez_clinica, COUNT(*) as cnt FROM silver.planilha_embriologia_combined GROUP BY 1").df()
    loc_comb_gb = conn.execute("SELECT fet_gravidez_bioquimica as gravidez_bioquimica, COUNT(*) as cnt FROM silver.planilha_embriologia_combined GROUP BY 1").df()
    loc_comb_nn = conn.execute("SELECT fet_no_nascidos as no_nascidos, COUNT(*) as cnt FROM silver.planilha_embriologia_combined GROUP BY 1").df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    ath_comb_res = pd.read_sql("SELECT fet_resultado as resultado, COUNT(*) as cnt FROM planilha_embriologia_combined GROUP BY 1", conn)
    ath_comb_gc = pd.read_sql("SELECT fet_gravidez_clinica as gravidez_clinica, COUNT(*) as cnt FROM planilha_embriologia_combined GROUP BY 1", conn)
    ath_comb_gb = pd.read_sql("SELECT fet_gravidez_bioquimica as gravidez_bioquimica, COUNT(*) as cnt FROM planilha_embriologia_combined GROUP BY 1", conn)
    ath_comb_nn = pd.read_sql("SELECT fet_n_nascidos as no_nascidos, COUNT(*) as cnt FROM planilha_embriologia_combined GROUP BY 1", conn)

print("--- Combined FET Resultado Distribution ---")
display(merge_dist(loc_comb_res, ath_comb_res, 'resultado'))

print("\n--- Combined FET Gravidez Clinica Distribution ---")
display(merge_dist(loc_comb_gc, ath_comb_gc, 'gravidez_clinica'))

print("\n--- Combined FET Gravidez Bioquimica Distribution ---")
display(merge_dist(loc_comb_gb, ath_comb_gb, 'gravidez_bioquimica'))

print("\n--- Combined FET No Nascidos Distribution ---")
display(merge_dist(loc_comb_nn, ath_comb_nn, 'no_nascidos'))


In [ ]:
# 6. Compare FET Quantitative Metrics in combined table
with duckdb.connect(DUCKDB_PATH, read_only=True) as conn:
    loc_comb_fet_sums = conn.execute('''
        SELECT 
            SUM(TRY_CAST(fet_no_nascidos AS DOUBLE)) as sum_fet_no_nascidos
        FROM silver.planilha_embriologia_combined
    ''').df()

with connect(region_name=ATHENA_REGION, work_group=ATHENA_WORKGROUP, schema_name=ATHENA_DB) as conn:
    ath_comb_fet_sums = pd.read_sql('''
        SELECT 
            SUM(TRY_CAST(fet_no_nascidos AS DOUBLE)) as sum_fet_no_nascidos
        FROM planilha_embriologia_combined
    ''', conn)

fet_comb_sum_cols = [
    ('Sum fet_no_nascidos', 'sum_fet_no_nascidos')
]

df_comb_fet_sums = pd.DataFrame({
    'Combined FET Metric': [c[0] for c in fet_comb_sum_cols],
    'Local (DuckDB)': [loc_comb_fet_sums.loc[0, c[1]] for c in fet_comb_sum_cols],
    'Athena (Prod)': [ath_comb_fet_sums.loc[0, c[1]] for c in fet_comb_sum_cols]
})
df_comb_fet_sums['Delta'] = df_comb_fet_sums['Local (DuckDB)'] - df_comb_fet_sums['Athena (Prod)']
df_comb_fet_sums['Match Rate %'] = np.where(
    df_comb_fet_sums['Delta'] == 0,
    100.0,
    (1.0 - (df_comb_fet_sums['Delta'].abs() / df_comb_fet_sums['Local (DuckDB)'])) * 100.0
)
print('--- Combined FET Quantitative Metric Aggregations ---')
print(df_comb_fet_sums.to_string(index=False))
